# Siamese Network for Person Re-Identification

GPU training notebook for the GitHub project. Heavy training is intended to run on Google Colab; the trained checkpoint can then be used for CPU inference locally.


In [1]:
!nvidia-smi


Sat Sep 19 10:37:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
%cd /content/Siamese-Network-for-Person-Re-Identification

!pip install -q -r requirements.txt

fatal: destination path 'Siamese-Network-for-Person-Re-Identification' already exists and is not an empty directory.
/content/Siamese-Network-for-Person-Re-Identification


In [5]:
!git clone https://github.com/parth1620/Person-Re-Id-Dataset

Cloning into 'Person-Re-Id-Dataset'...
remote: Enumerating objects: 12942, done.
remote: Counting objects: 100% (12942/12942), done.
remote: Compressing objects: 100% (12942/12942), done.
remote: Total 12942 (delta 0), reused 12942 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12942/12942), 27.68 MiB | 30.21 MiB/s, done.


In [15]:
DATA_ROOT ="/content/Siamese-Network-for-Person-Re-Identification/Person-Re-Id-Dataset"

# Expected:
# DATA_ROOT/train/train.csv
# DATA_ROOT/train/*.jpg

!find "$DATA_ROOT" -maxdepth 2 -type f | head -20

/content/Siamese-Network-for-Person-Re-Identification/Person-Re-Id-Dataset/train/0002_c1s1_000451_03.jpg
/content/Siamese-Network-for-Person-Re-Identification/Person-Re-Id-Dataset/train/0002_c1s1_000551_01.jpg
/content/Siamese-Network-for-Person-Re-Identification/Person-Re-Id-Dataset/train/0002_c1s1_000776_01.jpg
/content/Siamese-Network-for-Person-Re-Identification/Person-Re-Id-Dataset/train/0002_c1s1_000801_01.jpg
/content/Siamese-Network-for-Person-Re-Identification/Person-Re-Id-Dataset/train/0002_c1s1_069056_02.jpg
/content/Siamese-Network-for-Person-Re-Identification/Person-Re-Id-Dataset/train/0002_c1s2_000841_01.jpg
/content/Siamese-Network-for-Person-Re-Identification/Person-Re-Id-Dataset/train/0002_c1s2_050821_02.jpg
/content/Siamese-Network-for-Person-Re-Identification/Person-Re-Id-Dataset/train/0002_c1s2_050846_02.jpg
/content/Siamese-Network-for-Person-Re-Identification/Person-Re-Id-Dataset/train/0002_c1s2_064446_01.jpg
/content/Siamese-Network-for-Person-Re-Identification/P

In [16]:
!find "$DATA_ROOT" -type f | grep -E '\.(csv|CSV)$'

/content/Siamese-Network-for-Person-Re-Identification/Person-Re-Id-Dataset/train.csv


## 5. Train on GPU


In [8]:
!python -m src.train --data-root "$DATA_ROOT" --device cuda --seed 42



model.safetensors: downloading bytes:   0% 0.00/21.4M [00:00<?, ?B/s]
model.safetensors: downloading bytes: 100% 20.2M/20.2M [00:00<00:00, 48.4MB/s, 2.00MB/s  ]
model.safetensors: reconstructing file: 100% 21.4M/21.4M [00:00<00:00, 51.1MB/s, 2.12MB/s  ]
Device: cuda
Training triplets: 3200
Validation triplets: 800
Epoch 01/15 | train_loss=0.68022 | valid_loss=0.59858
Saved best checkpoint.
Epoch 02/15 | train_loss=0.30654 | valid_loss=0.40532
Saved best checkpoint.
Epoch 03/15 | train_loss=0.19084 | valid_loss=0.29461
Saved best checkpoint.
Epoch 04/15 | train_loss=0.12777 | valid_loss=0.27563
Saved best checkpoint.
Epoch 05/15 | train_loss=0.09281 | valid_loss=0.17266
Saved best checkpoint.
Epoch 06/15 | train_loss=0.06847 | valid_loss=0.23608
Epoch 07/15 | train_loss=0.06197 | valid_loss=0.21033
Epoch 08/15 | train_loss=0.07784 | valid_loss=0.21952
Epoch 09/15 | train_loss=0.06291 | valid_loss=0.26329
Epoch 10/15 | train_loss=0.05581 | valid_loss=0.18639
Epoch 11/15 | train_loss=0.0

In [11]:
!mkdir -p outputs/results

In [17]:
# 8. Check trained model
!ls -lh checkpoints/

total 19M
-rw-r--r-- 1 root root 19M Sep 19 10:45 best_model.pt


## 6. Copy the best checkpoint to Google Drive


In [12]:
!mkdir -p "/content/drive/MyDrive/siamese-person-reid/checkpoints"
!cp checkpoints/best_model.pt "/content/drive/MyDrive/siamese-person-reid/checkpoints/best_model.pt"


## 7. Generate the embedding database


In [18]:
!python -m src.embeddings \
  --checkpoint checkpoints/best_model.pt \
  --triplet-csv "$DATA_ROOT/train.csv" \
  --image-dir "$DATA_ROOT/train" \
  --output outputs/results/database.csv \
  --device cuda

100% 3285/3285 [00:34<00:00, 95.99it/s] 
Saved 3285 embeddings to outputs/results/database.csv


In [19]:
!python -m src.evaluate \
  --database outputs/results/database.csv \
  --top-k 1 5 10


Queries evaluated: 3285
Recall@1: 0.4088
Recall@5: 0.6770
Recall@10: 0.7653
